# 01 — Dataset Preparation

This notebook prepares the dataset and creates the 5-fold cross-validation manifest for DenseNet121.

> **Educational Use Only** — This is not a medical diagnostic tool.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Clone or Update the Repository

In [ ]:
import os

# --- Configure these paths to match your Google Drive layout ---
DRIVE_BASE = '/content/drive/MyDrive'
REPO_DIR = os.path.join(DRIVE_BASE, 'histology-ai-classification')
REPO_URL = 'https://github.com/<your-org>/histology-ai-classification.git'

if os.path.exists(REPO_DIR):
    print('Repository already exists — pulling latest changes...')
    os.chdir(REPO_DIR)
    !git pull
else:
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print('Working directory:', os.getcwd())

## Step 3 — Install Dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

## Step 4 — Reconstruct the 22-Class Dataset

The raw NuInsSeg dataset must be downloaded from the official source and placed at a path accessible here.
See [NuInsSeg on Zenodo](https://doi.org/10.5281/zenodo.10518852).

In [ ]:
# --- Set the path to your NuInsSeg download ---
NUINSSEG_SOURCE = os.path.join(DRIVE_BASE, 'NuInsSeg')  # Adjust if needed
DEST_ROOT = 'data/raw/nuinsseg_human_22_original'

import subprocess, sys
result = subprocess.run([
    sys.executable, 'scripts/create_original_22_class_dataset.py',
    '--source-root', NUINSSEG_SOURCE,
    '--destination-root', DEST_ROOT,
    '--overwrite'
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 5 — Exploratory Data Analysis

In [ ]:
!python -m src.data.explore_dataset --data-dir data/raw/nuinsseg_human_22_original

## Step 6 — Verify the Manifest

In [ ]:
import pandas as pd
df = pd.read_csv('data/manifests/original_22_dataset_manifest.csv')
print(f'Total images: {len(df)}')
print(f'Classes: {df["class_name"].nunique()}')
print('Distribution:')
print(df['class_name'].value_counts().sort_index())

## Step 7 — Create DenseNet121 Folds

In [ ]:
!python -m src.data.create_folds \
    --manifest-path data/manifests/original_22_dataset_manifest.csv \
    --output-path data/manifests/densenet121_folds.csv \
    --num-folds 5 \
    --seed 42

## Step 8 — Verify Fold Distribution

In [ ]:
df_folds = pd.read_csv('data/manifests/densenet121_folds.csv')
print(f'Total images: {len(df_folds)}')
print(f'Unique IDs: {df_folds["image_id"].nunique()}')
print('Images per fold:')
print(df_folds['fold'].value_counts().sort_index())
print('\nClass x Fold distribution:')
print(df_folds.groupby(['class_name', 'fold']).size().unstack(fill_value=0))